In [1]:
# 기존 디렉토리를 삭제
!rm -r aclImdb

# IMDB 리뷰 데이터셋 다운로드 및 압축 해제
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

# 비지도학습 데이터셋(unsup)을 삭제
!rm -r aclImdb/train/unsup

import os, pathlib, shutil, random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

batch_size = 32

# 데이터 디렉토리 경로 설정
base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"  # 검증 데이터 경로
train_dir = base_dir / "train"  # 학습 데이터 경로

# 학습 데이터를 학습/검증 데이터로 분리
for category in ("neg", "pos"):
    os.makedirs(val_dir / category)  # 검증 데이터 폴더 생성
    files = os.listdir(train_dir / category)  # 해당 폴더의 파일 목록 가져오기
    random.Random(1337).shuffle(files)  # 고정된 랜덤 시드로 파일 섞기
    num_val_samples = int(0.2 * len(files))  # 20%를 검증 데이터로 설정
    val_files = files[-num_val_samples:]  # 검증 데이터 파일 선택
    for fname in val_files:  # 파일 이동
        shutil.move(train_dir / category / fname,  val_dir / category / fname)

# 텍스트 데이터셋 로드
train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size  # 학습 데이터셋
)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size  # 검증 데이터셋
)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size  # 테스트 데이터셋
)
text_only_train_ds = train_ds.map(lambda x, y: x)  # 텍스트 데이터만 추출

# 텍스트 벡터화 설정
max_length = 600  # 최대 문장 길이
max_tokens = 20000  # 최대 단어 수
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",  # 정수 인코딩
    output_sequence_length=max_length,  # 고정된 시퀀스 길이
)
text_vectorization.adapt(text_only_train_ds)  # 학습 데이터로 벡터화 레이어 학습

# 텍스트 데이터를 벡터화된 데이터로 변환
int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)  # 병렬 처리
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

# Transformer Encoder 레이어 정의
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim  # 멀티헤드 어텐션 레이어
        )
        self.dense_proj = keras.Sequential([  # 피드포워드 네트워크
            layers.Dense(dense_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm_1 = layers.LayerNormalization()  # 첫 번째 LayerNorm
        self.layernorm_2 = layers.LayerNormalization()  # 두 번째 LayerNorm

    def build(self, input_shape):
        # MultiHeadAttention 및 하위 레이어 빌드
        self.attention.build(input_shape, input_shape)
        self.dense_proj.build(input_shape)
        self.layernorm_1.build(input_shape)
        self.layernorm_2.build(input_shape)
        super().build(input_shape)  # 상위 클래스 빌드 호출

    def call(self, inputs, mask=None):
        if mask is not None:
            mask = mask[:, tf.newaxis, :]  # 마스크 차원 확장
        attention_output = self.attention(inputs, inputs, attention_mask=mask)  # 어텐션 계산
        proj_input = self.layernorm_1(inputs + attention_output)  # 첫 번째 LayerNorm 적용
        proj_output = self.dense_proj(proj_input)  # 피드포워드 네트워크 적용
        return self.layernorm_2(proj_input + proj_output)  # 두 번째 LayerNorm 적용

    def get_config(self):
        # 레이어 설정 반환
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dense_dim": self.dense_dim,
        })
        return config

# 모델 구성
vocab_size = 20000  # 단어 집합 크기
embed_dim = 256  # 임베딩 차원
num_heads = 2  # 어텐션 헤드 수
dense_dim = 32  # 피드포워드 네트워크의 밀집 레이어 크기

inputs = keras.Input(shape=(None,), dtype="int64")  # 입력 정의
x = layers.Embedding(vocab_size, embed_dim)(inputs)  # 임베딩 레이어
x = TransformerEncoder(embed_dim, dense_dim, num_heads)(x)  # Transformer Encoder 레이어
x = layers.GlobalMaxPooling1D()(x)  # Global Max Pooling
x = layers.Dropout(0.5)(x)  # 드롭아웃 적용
outputs = layers.Dense(1, activation="sigmoid")(x)  # 출력 레이어 (시그모이드 활성화)
model = keras.Model(inputs, outputs)  # 모델 생성
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])  # 모델 컴파일
model.summary()  # 모델 요약 출력

# 모델 학습 및 저장
callbacks = [
    keras.callbacks.ModelCheckpoint("transformer_encoder.keras", save_best_only=True)
]
model.fit(int_train_ds, validation_data=int_val_ds, epochs=20, callbacks=callbacks)
model = keras.models.load_model(
    "transformer_encoder.keras", custom_objects={"TransformerEncoder": TransformerEncoder}
)
print(f"테스트 정확도: {model.evaluate(int_test_ds)[1]:.3f}")  # 테스트 정확도 출력


rm: cannot remove 'aclImdb': No such file or directory
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  46.2M      0  0:00:01  0:00:01 --:--:-- 46.2M
Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, None, 256)      │     5,120,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder             │ (None, None, 256)      │       543,776 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 256)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,664,033 (21.61 MB)

 Trainable params: 5,664,033 (21.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 45s 61ms/step - accuracy: 0.6257 - loss: 0.7238 - val_accuracy: 0.8730 - val_loss: 0.2926
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 39s 62ms/step - accuracy: 0.8883 - loss: 0.2655 - val_accuracy: 0.8774 - val_loss: 0.3086
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 61ms/step - accuracy: 0.9525 - loss: 0.1214 - val_accuracy: 0.8728 - val_loss: 0.4514
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 42s 62ms/step - accuracy: 0.9803 - loss: 0.0549 - val_accuracy: 0.8640 - val_loss: 0.5072
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 61ms/step - accuracy: 0.9894 - loss: 0.0310 - val_accuracy: 0.8562 - val_loss: 0.7212
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 36s 58ms/step - accuracy: 0.9875 - loss: 0.0354 - val_accuracy: 0.8650 - val_loss: 0.7428
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 43s 61ms/step - accuracy: 0.9923 - loss: 0.0238 - val_accuracy: 0.8510 - val_loss: 0.7608
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 38s 61ms/step - accuracy: 0.9941 - loss: 0.0156 - 

In [1]:
# Positional Embedding 추가
# 기존 디렉토리를 삭제
!rm -r aclImdb

# IMDB 리뷰 데이터셋 다운로드 및 압축 해제
!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xf aclImdb_v1.tar.gz

# 비지도학습 데이터셋(unsup)을 삭제
!rm -r aclImdb/train/unsup

import os, pathlib, shutil, random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

batch_size = 32

# 데이터 디렉토리 경로 설정
base_dir = pathlib.Path("aclImdb")
val_dir = base_dir / "val"  # 검증 데이터 경로
train_dir = base_dir / "train"  # 학습 데이터 경로

# 학습 데이터를 학습/검증 데이터로 분리
for category in ("neg", "pos"):
    os.makedirs(val_dir / category)  # 검증 데이터 폴더 생성
    files = os.listdir(train_dir / category)  # 해당 폴더의 파일 목록 가져오기
    random.Random(1337).shuffle(files)  # 고정된 랜덤 시드로 파일 섞기
    num_val_samples = int(0.2 * len(files))  # 20%를 검증 데이터로 설정
    val_files = files[-num_val_samples:]  # 검증 데이터 파일 선택
    for fname in val_files:  # 파일 이동
        shutil.move(train_dir / category / fname,  val_dir / category / fname)

# 텍스트 데이터셋 로드
train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size  # 학습 데이터셋
)
val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size  # 검증 데이터셋
)
test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size  # 테스트 데이터셋
)
text_only_train_ds = train_ds.map(lambda x, y: x)  # 텍스트 데이터만 추출

# 텍스트 벡터화 설정
max_length = 600  # 최대 문장 길이
max_tokens = 20000  # 최대 단어 수
text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",  # 정수 인코딩
    output_sequence_length=max_length,  # 고정된 시퀀스 길이
)
text_vectorization.adapt(text_only_train_ds)  # 학습 데이터로 벡터화 레이어 학습

# 텍스트 데이터를 벡터화된 데이터로 변환
int_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)  # 병렬 처리
int_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)
int_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y),
    num_parallel_calls=4)

# Transformer Encoder with Positional Embedding
class TransformerEncoder(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, max_length=600, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.dense_dim = dense_dim
        self.num_heads = num_heads
        self.max_length = max_length
        self.attention = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim  # 멀티헤드 어텐션 레이어
        )
        self.dense_proj = keras.Sequential([  # 피드포워드 네트워크
            layers.Dense(dense_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm_1 = layers.LayerNormalization()  # 첫 번째 LayerNorm
        self.layernorm_2 = layers.LayerNormalization()  # 두 번째 LayerNorm

        # Positional Embedding
        self.position_embedding = layers.Embedding(
            input_dim=max_length, output_dim=embed_dim
        )

    def build(self, input_shape):
        seq_length = input_shape[1]  # 시퀀스 길이
        self.position_embedding.build((seq_length,))

        # MultiHeadAttention 및 서브 레이어 빌드
        self.attention.build(
            query_shape=(input_shape[0], seq_length, self.embed_dim),
            key_shape=(input_shape[0], seq_length, self.embed_dim),
            value_shape=(input_shape[0], seq_length, self.embed_dim),
        )
        self.dense_proj.build((input_shape[0], seq_length, self.embed_dim))
        self.layernorm_1.build((input_shape[0], seq_length, self.embed_dim))
        self.layernorm_2.build((input_shape[0], seq_length, self.embed_dim))
        super().build(input_shape)

    def call(self, inputs, mask=None):
        # inputs의 shape: (batch_size, seq_length, embed_dim)
        seq_length = tf.shape(inputs)[1]
        positions = tf.range(start=0, limit=seq_length, delta=1)  # (seq_length,)
        embedded_positions = self.position_embedding(positions)  # (seq_length, embed_dim)
        inputs += embedded_positions  # Position 정보를 더하기

        if mask is not None:
            mask = mask[:, tf.newaxis, :]  # 마스크 차원 확장
        attention_output = self.attention(inputs, inputs, attention_mask=mask)  # 어텐션 계산
        proj_input = self.layernorm_1(inputs + attention_output)  # 첫 번째 LayerNorm 적용
        proj_output = self.dense_proj(proj_input)  # 피드포워드 네트워크 적용
        return self.layernorm_2(proj_input + proj_output)  # 두 번째 LayerNorm 적용

    def get_config(self):
        # 레이어 설정 반환
        config = super().get_config()
        config.update({
            "embed_dim": self.embed_dim,
            "num_heads": self.num_heads,
            "dense_dim": self.dense_dim,
            "max_length": self.max_length,
        })
        return config

# 모델 구성
vocab_size = 20000  # 단어 집합 크기
embed_dim = 256  # 임베딩 차원
num_heads = 2  # 어텐션 헤드 수
dense_dim = 32  # 피드포워드 네트워크의 밀집 레이어 크기

inputs = keras.Input(shape=(None,), dtype="int64")  # 입력 정의
x = layers.Embedding(vocab_size, embed_dim)(inputs)  # 임베딩 레이어
x = TransformerEncoder(embed_dim, dense_dim, num_heads, max_length=max_length)(x)  # Transformer Encoder 레이어
x = layers.GlobalMaxPooling1D()(x)  # Global Max Pooling
x = layers.Dropout(0.5)(x)  # 드롭아웃 적용
outputs = layers.Dense(1, activation="sigmoid")(x)  # 출력 레이어 (시그모이드 활성화)
model = keras.Model(inputs, outputs)  # 모델 생성
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])  # 모델 컴파일
model.summary()  # 모델 요약 출력

# 모델 학습 및 저장
callbacks = [
    keras.callbacks.ModelCheckpoint("transformer_encoder.keras", save_best_only=True)
]
model.fit(int_train_ds, validation_data=int_val_ds, epochs=20, callbacks=callbacks)
model = keras.models.load_model(
    "transformer_encoder.keras", custom_objects={"TransformerEncoder": TransformerEncoder}
)
print(f"테스트 정확도: {model.evaluate(int_test_ds)[1]:.3f}")  # 테스트 정확도 출력

rm: cannot remove 'aclImdb': No such file or directory
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 80.2M  100 80.2M    0     0  3162k      0  0:00:25  0:00:25 --:--:-- 8530k
Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, None, 256)      │     5,120,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder             │ (None, None, 256)      │       697,376 │
│ (TransformerEncoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 256)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,817,633 (22.19 MB)

 Trainable params: 5,817,633 (22.19 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 46s 60ms/step - accuracy: 0.5983 - loss: 0.7517 - val_accuracy: 0.8462 - val_loss: 0.3447
Epoch 2/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 37s 60ms/step - accuracy: 0.8785 - loss: 0.2887 - val_accuracy: 0.8614 - val_loss: 0.4149
Epoch 3/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 40s 58ms/step - accuracy: 0.9469 - loss: 0.1375 - val_accuracy: 0.8626 - val_loss: 0.4946
Epoch 4/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 39s 55ms/step - accuracy: 0.9762 - loss: 0.0685 - val_accuracy: 0.8526 - val_loss: 0.5785
Epoch 5/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 34s 55ms/step - accuracy: 0.9857 - loss: 0.0376 - val_accuracy: 0.8578 - val_loss: 0.6559
Epoch 6/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 34s 55ms/step - accuracy: 0.9870 - loss: 0.0382 - val_accuracy: 0.8550 - val_loss: 0.7638
Epoch 7/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 41s 56ms/step - accuracy: 0.9876 - loss: 0.0342 - val_accuracy: 0.8576 - val_loss: 0.6969
Epoch 8/20
625/625 ━━━━━━━━━━━━━━━━━━━━ 35s 55ms/step - accuracy: 0.9916 - loss: 0.0266 - 